# Does one model for many species beat a model per species?

Spring bloom in temperate fruit trees is governed by a shared physiology: a chilling
requirement satisfied over winter, followed by heat accumulation that pushes the buds to
open. Because apple, pear and the stone fruits all run on this same machinery, a single model
trained on *all* of them at once might learn the common signal better than many small
per-species models — and could lend its knowledge to species that have only a handful of
recorded years. The risk is the opposite: pooling could blur away the genuine differences
between species and end up worse than bespoke models.

This notebook puts that question to the test on ground-truth phenology from the PEP725 network
of European fruit trees (apple, pear, peach, plum and cherry), using ERA5 mean temperature and
photoperiod as the daily drivers. We compare four predictors:

| model | what it knows about species |
|---|---|
| **per-species mean** | nothing — predicts that species' average bloom day |
| **per-species LSTM** | nothing — a separate network is fit for each species |
| **multi-species LSTM, one-hot species** | species identity as an arbitrary label |
| **multi-species LSTM, phylogenetic embedding** | species identity as a point in a tree-of-life space |

We look at two situations. First, an ordinary **year split** where every species has plenty
of history and we test on its most recent years. Second, a **few-shot** setting where we
pretend a species is newly added: the multi-species models see all the *other* species in
full but only `x` records of the newcomer, and we ask how quickly each model adapts as `x`
grows from zero.

Throughout we score models by **mean absolute error averaged over species** (compute the MAE
within each species, then average those). Sample counts are wildly uneven — apple and cherry
dominate by volume — and a plain pooled MAE would mostly report how well we do on those two.
Averaging per species instead asks every species to count the same, which is the question we
actually care about.

## Configuration and imports

In [ ]:
import json, math
from functools import reduce
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from pysephone.dataset.observations import Observations
from pysephone.dataset.dataset import Dataset
from pysephone.dataset.registry import REGISTRY
from pysephone.dataset.util.calendar import Calendar
from pysephone.dataset.util.openmeteo import OpenMeteoFeatures
from pysephone.dataset.util.daylength import DaylengthFeatures
from pysephone.dataset.util.phylogeny import PhylogenyFeatures
from pysephone.benchmarks.bloombench import config as bb_config
from pysephone.models.base import ModelException
from pysephone.models.mean import MeanModel
from pysephone.models.lstm import LSTMModel
from pysephone.models.lstm_ctx import OneHotSpeciesLSTMModel, PhylogeneticLSTMModel
from pysephone.paths import get_data_root
from pysephone.constants import (
    KEY_DATA_SOURCE, KEY_SPECIES_ID, KEY_OBS_TYPE, KEY_OBSERVATIONS, KEYS_INDEX,
)

# Registry key -> the observation key that marks bloom for that dataset.
DATASETS = {
    'PEP725_Apple':           'BBCH_60',
    'PEP725_Pear':            'BBCH_60',
    'PEP725_Peach':           'BBCH_60',   # year-split: excluded (almost no post-2004 records); kept for few-shot
    # 'PEP725_Almond':          'BBCH_60',   # ~150 records — too few to fit a per-species LSTM
    # 'PEP725_Apricot':         'BBCH_60',   # ~190 records — too few to fit a per-species LSTM
    'PEP725_Plum':            'BBCH_60',
    'PEP725_Cherry':          'BBCH_60',   # full European sweet cherry (no GMU here, so no overlap to drop)
    # 'GMU_Cherry_Japan_Y':     'gmu_0',     # Somei-yoshino
    # 'GMU_Cherry_Japan_S':     'gmu_0',     # Sargent cherry
    # 'GMU_Cherry_Switzerland': 'gmu_1',     # Swiss sweet cherry
    # 'GMU_Cherry_South_Korea': 'gmu_2',     # Korean king cherry
}

# Species kept out of the year split because they lack a usable recent test fold,
# but still loaded (they serve as few-shot hold-outs and feed the phylogeny).
YEAR_SPLIT_DROP = {'PEP725_Peach'}

TEMP_KEY  = 'temperature_2m_mean'          # Open-Meteo serves ERA5 reanalysis
DATA_KEYS = [TEMP_KEY, 'daylength']
# Each dataset records bloom under its own key; we relabel them to one shared
# event so samples from different species can sit in the same training batch.
BLOOM_KEY = 'bloom'

SEASON_START  = bb_config.SEASON_START      # '10-01'
SEASON_LENGTH = bb_config.SEASON_LENGTH     # 365
# Single train/test boundary shared by all species (train < year <= test). Placed
# a touch before the ~75% mark so the data-rich species keep healthy test folds.
CUTOFF_YEAR   = 2009

X_GRID        = [0, 5, 10, 25, 50]
SEED          = 0
DEVICE        = bb_config.torch_device()
FORCE_RETRAIN = False

# Flip to True for a fast end-to-end check on a handful of species.
SMOKE = False
if SMOKE:
    DATASETS = {k: DATASETS[k] for k in
                ['PEP725_Apple', 'PEP725_Cherry', 'PEP725_Plum']}
    X_GRID = [0, 25]

# In the raw GMU data the Korean cherry carries species_id 0, the same id as the
# Japanese Somei-yoshino. Give it its own id so each dataset stays one species.
SK_KEY, SK_SPECIES_ID = 'GMU_Cherry_South_Korea', 9

# The GMU source ships no scientific names; supply them so the cherries can be
# placed in the tree of life alongside the named PEP725 species.
GMU_NAMES = {
    ('GMU_cherry', 0):             'Prunus yedoensis',
    ('GMU_cherry', 1):             'Prunus sargentii',
    ('GMU_cherry', 5):             'Prunus avium',
    ('GMU_cherry', SK_SPECIES_ID): 'Prunus yedoensis',
}

ART = get_data_root() / 'outputs' / 'multispecies'
ART.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED); np.random.seed(SEED)
print(f'device={DEVICE}  datasets={len(DATASETS)}  artifacts={ART}')

## Assembling the data

Each dataset is one species. We pull them from the registry, relabel every dataset's bloom
record to one shared event so samples from different species can share a training batch, and
merge them into a single collection with two daily drivers attached — ERA5 mean temperature
(via Open-Meteo) and an analytic daylength — over an October-to-October season. Scientific
names are carried through for the phylogeny step. (The loader also keeps guards for the GMU
cherry datasets — distinct species ids and hand-supplied names — for runs that include them.)

In [ ]:
def load_merged():
    obs_list, names, ds_species = [], {}, {}
    for key, bloom in DATASETS.items():
        obs = REGISTRY[key]()
        df = obs.df_y.reset_index()
        df = df[df[KEY_OBS_TYPE] == bloom].copy()      # keep only the bloom stage
        df[KEY_OBS_TYPE] = BLOOM_KEY                    # one shared event key for all species
        if key == SK_KEY:
            df[KEY_SPECIES_ID] = SK_SPECIES_ID          # break the id collision with Japan
        df = df.set_index(list(KEYS_INDEX) + [KEY_OBS_TYPE])
        obs = Observations(df, obs.df_y_loc, species_names=obs.species_names)
        ds_species[key] = obs.species[0]                # one (src, species_id) per dataset
        names.update(obs.species_names)                 # PEP725 ships names; GMU does not
        obs_list.append(obs)
    present = set(ds_species.values())
    names.update({k: v for k, v in GMU_NAMES.items() if k in present})
    merged = reduce(Observations.merge, obs_list)
    merged = Observations(merged.df_y, merged.df_y_loc, species_names=names)
    return merged, ds_species

merged_obs, DS_SPECIES = load_merged()
SPECIES_LABEL = {sp: key for key, sp in DS_SPECIES.items()}
ALL_SPECIES   = sorted(DS_SPECIES.values())                       # every loaded species
YEAR_SPECIES  = [sp for sp in ALL_SPECIES                         # species used in the year split
                 if SPECIES_LABEL[sp] not in YEAR_SPLIT_DROP]

calendar  = Calendar(default_start=SEASON_START, default_length=SEASON_LENGTH)
openmeteo = OpenMeteoFeatures(calendar, data_keys=[TEMP_KEY], step='daily')
daylength = DaylengthFeatures(calendar)
full = Dataset(merged_obs, calendar=calendar, feature_providers=[openmeteo, daylength])
full.download_features(verbose=True)     # ERA5 is mostly cached; daylength is analytic
print(f'{len(full)} samples, {len(ALL_SPECIES)} species '
      f'({len(YEAR_SPECIES)} in the year split)')

We hold out the most recent years for testing, so every model is judged on predicting the
future from the past. The boundary is a **single cut-off year shared by all species**, not a
per-species one: the multi-species models train on one pooled set and test on another, and
only a common boundary keeps that pooled split temporally clean and comparable to the
per-species splits. It sits just before the three-quarter mark so the data-rich species keep
healthy test folds; species counts still come out somewhat uneven, which is the price of a
consistent boundary. Peach is left out of the year split entirely — almost all of its records
predate 2005, so no recent cut-off gives it a meaningful test fold — but it stays in the data
as a few-shot hold-out below.

In [ ]:
# Year split: restrict to the year-split species, then cut every one of them at
# the same CUTOFF_YEAR. A single boundary keeps the pooled multi-species split
# temporally clean and directly comparable to the per-species splits.
year_full  = full.select_species(YEAR_SPECIES)
years_all  = sorted(set(year_full.years))
pool_train = year_full.select_years([y for y in years_all if y <  CUTOFF_YEAR])
pool_test  = year_full.select_years([y for y in years_all if y >= CUTOFF_YEAR])
frac = len(pool_train) / max(1, len(pool_train) + len(pool_test))
print(f'cut-off {CUTOFF_YEAR}: train={len(pool_train)}  test={len(pool_test)}  ({frac:.0%} train)')

summary = []
for sp in YEAR_SPECIES:
    yrs = sorted(set(full.select_species(sp).years))
    summary.append(dict(
        species=SPECIES_LABEL[sp], name=merged_obs.get_species_name(*sp),
        n_train=len(pool_train.select_species(sp)),
        n_test=len(pool_test.select_species(sp)),
        years=f'{min(yrs)}-{max(yrs)}'))
pd.DataFrame(summary).set_index('species')

## Reading off the bloom date

When the datasets were merged, each one's bloom record was relabelled to a single shared
event, so the prediction target is simply that one date for every species.

In [ ]:
def target_fn(sample):
    return sample[KEY_OBSERVATIONS][BLOOM_KEY]

_s = next(pool_train.iter_items())
print(SPECIES_LABEL[(_s[KEY_DATA_SOURCE], _s[KEY_SPECIES_ID])], '->', target_fn(_s))

## A phylogenetic coordinate for each species

To let the model exploit relatedness, we resolve every species against the Open Tree of Life,
read off the branch-length (patristic) distances between them, and embed those distances in a
few continuous coordinates with classical multidimensional scaling. The result and the
distance matrix are cached so the run is reproducible and works offline afterwards.

In [ ]:
PHYLO_NPZ = ART / 'phylogeny.npz'

if PHYLO_NPZ.exists() and not FORCE_RETRAIN:
    d = np.load(PHYLO_NPZ, allow_pickle=True)
    phylo_keys = [(str(s), int(i)) for s, i in d['keys']]
    mds_full, dist_mat = d['mds'], d['dist']
else:
    _phylo = PhylogenyFeatures(k_embed=10, output=['mds']).fit(merged_obs)
    phylo_keys = [(str(s), int(i)) for s, i in _phylo.species_keys]
    mds_full   = np.asarray(_phylo.mds_coords)
    dist_mat   = np.asarray(_phylo.distance_matrix)
    np.savez(PHYLO_NPZ, keys=np.array(phylo_keys, dtype=object), mds=mds_full, dist=dist_mat)

assert phylo_keys == ALL_SPECIES, 'phylogeny species must match the dataset species'
print('resolved', len(phylo_keys), 'species in the tree of life')

How many coordinates do we actually need? Classical MDS on `N` species can produce at
most `N-1` informative axes, and the variance each axis carries is its eigenvalue in the
double-centred distance matrix. These species are almost all *Prunus*, with a single *Malus*
(apple) and *Pyrus* (pear), so the tree is shallow and a couple of axes should capture nearly
all of the structure. We keep the fewest axes that reach 90% of the variance — and deliberately
keep that number small relative to the LSTM's hidden width, so the species signal informs the
readout without swamping the weather representation.

In [ ]:
N = dist_mat.shape[0]
H = np.eye(N) - np.ones((N, N)) / N
B = -0.5 * H @ (dist_mat ** 2) @ H
ev = np.sort(np.linalg.eigvalsh((B + B.T) / 2))[::-1]
cum = np.cumsum(np.clip(ev, 0, None)) / np.clip(ev, 0, None).sum()

K_PHYLO = max(2, min(int(np.searchsorted(cum, 0.90)) + 1, N - 1, mds_full.shape[1]))
mds_coords = mds_full[:, :K_PHYLO]

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(range(1, len(cum) + 1), cum, 'o-')
ax.axhline(0.90, ls='--', c='grey'); ax.axvline(K_PHYLO, ls='--', c='crimson')
ax.set_xlabel('MDS dimensions kept'); ax.set_ylabel('cumulative variance'); ax.set_ylim(0, 1.02)
ax.set_title(f'Keeping K = {K_PHYLO} phylogenetic dimensions'); plt.tight_layout(); plt.show()
print('cumulative variance by dimension:', np.round(cum, 3))

In [ ]:
# Where the species land in the first two phylogenetic dimensions.
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.scatter(mds_full[:, 0], mds_full[:, 1])
for sp, xy in zip(phylo_keys, mds_full[:, :2]):
    ax.annotate(merged_obs.get_species_name(*sp), xy, fontsize=8,
                xytext=(4, 2), textcoords='offset points')
ax.set_xlabel('MDS-1'); ax.set_ylabel('MDS-2')
ax.set_title('Species arranged by phylogenetic distance'); plt.tight_layout(); plt.show()

## Scoring

For a fitted model we predict every sample, take the absolute error in days, average within each species, and then average across species. We keep the sample-weighted error alongside it to show how much the headline number shifts once every species counts equally.

In [ ]:
def evaluate(model, dataset):
    per = {}
    for s in dataset.iter_items():
        pred, _ = model.predict(s)
        tgt = np.datetime64(target_fn(s), 'D')
        err = float(abs((np.datetime64(pred, 'D') - tgt) / np.timedelta64(1, 'D')))
        per.setdefault((s[KEY_DATA_SOURCE], s[KEY_SPECIES_ID]), []).append(err)
    per_mae = {sp: float(np.mean(v)) for sp, v in per.items()}
    total   = sum(sum(v) for v in per.values())
    n       = sum(len(v) for v in per.values())
    return dict(macro=float(np.mean(list(per_mae.values()))) if per_mae else float('nan'),
                micro=total / n if n else float('nan'),
                per_species=per_mae)

## One hyperparameter search, reused everywhere

We will train many networks across the two experiments, so tuning each one is out of the
question. Instead we run a single **random search** over four axes — batch size, hidden width,
learning rate, and the number of training epochs — sampling a handful of configurations,
scoring each by macro-MAE on the most recent training years, and keeping the best. Depth is
held fixed. That one configuration is then reused for every LSTM variant and every split: the
variants share the same backbone, and tuning a network on the few labelled examples of a
held-out species would only overfit the tiny validation slice. The tuned epoch count is an
**upper bound** — training stops early once a held-out validation slice stops improving (and
that slice is skipped when a training set is too small to spare one). The chosen settings are
cached.

In [ ]:
FEATURE_STATS = pool_train.compute_feature_stats(verbose=False)
HP_JSON      = ART / 'lstm_hp.json'
NUM_LAYERS   = 2                              # fixed; the search covers the four axes below
N_HPO_TRIALS = 4 if SMOKE else 16
_EPOCH_LO, _EPOCH_HI = (10, 30) if SMOKE else (40, 1000)   # num_epochs sampled in this range

# Early stopping turns num_epochs into an upper bound: training runs up to that many
# epochs but stops once the held-out validation stops improving.
ES_PATIENCE, ES_VAL_PERIOD, ES_MIN_DELTA = 5, 10, 1e-3
ES_MIN_SAMPLES = 50                           # below this, skip the val split and just train

def _fit_torch(model, ds, *, num_epochs, batch_size, lr):
    vp = ES_VAL_PERIOD if len(ds) >= ES_MIN_SAMPLES else None   # too few samples to hold out
    model, _ = type(model).fit(
        target_fn=target_fn, dataset=ds, model=model,
        optimizer_kwargs=dict(lr=lr, weight_decay=1e-4),
        num_epochs=num_epochs, batch_size=batch_size,
        val_period=vp, early_stopping=vp is not None,
        early_stopping_patience=ES_PATIENCE, early_stopping_min_delta=ES_MIN_DELTA,
        device=DEVICE, seed=SEED, verbose=True)
    return model

def _year_holdout(ds, frac=0.2):
    yrs = sorted(set(ds.years))
    cut = yrs[max(1, int(len(yrs) * (1 - frac)))]
    return (ds.select_years([y for y in yrs if y < cut]),
            ds.select_years([y for y in yrs if y >= cut]))

def _sample_hp(rng):
    return dict(
        batch_size    = int(rng.choice([32, 64, 128, 256, 512])),
        hidden_size   = int(rng.choice([32, 64, 128])),
        learning_rate = float(10 ** rng.uniform(-4, -2)),            # log-uniform 1e-4 .. 1e-2
        num_epochs    = int(rng.integers(_EPOCH_LO // 10, _EPOCH_HI // 10 + 1)) * 10,
    )

def select_hyperparameters(n_trials=N_HPO_TRIALS):
    if HP_JSON.exists() and not FORCE_RETRAIN:
        return json.loads(HP_JSON.read_text())
    trn, val = _year_holdout(pool_train)
    rng = np.random.default_rng(SEED)
    best, best_mae = None, math.inf
    for t in range(n_trials):
        hp = _sample_hp(rng)
        print(f"trial {t:2d}: bs={hp['batch_size']:<4d} hidden={hp['hidden_size']:<4d} "
              f"lr={hp['learning_rate']:.1e} epochs<={hp['num_epochs']}")
        m = LSTMModel(data_keys=DATA_KEYS, feature_statistics=FEATURE_STATS,
                      hidden_size=hp['hidden_size'], num_layers=NUM_LAYERS)
        m = _fit_torch(m, trn, num_epochs=hp['num_epochs'],
                       batch_size=hp['batch_size'], lr=hp['learning_rate'])
        mae = evaluate(m, val)['macro']
        marker = '  *' if mae < best_mae else ''
        print(f"  -> val macro-MAE {mae:.2f}{marker}")
        if mae < best_mae:
            best, best_mae = hp, mae
    HP_JSON.write_text(json.dumps(best))
    return best

BEST_HP = select_hyperparameters()
print('chosen hyperparameters:', BEST_HP)

## Building and persisting the models

Training is slow, so every fitted model is saved and reloaded by name on subsequent runs. The two multi-species models always know about all eleven species: the one-hot model reserves a slot per species, while the phylogenetic model is handed each species' coordinates — including for a species it has never trained on.

In [ ]:
def make_per_species_lstm():
    return LSTMModel(data_keys=DATA_KEYS, feature_statistics=FEATURE_STATS,
                     hidden_size=BEST_HP['hidden_size'], num_layers=NUM_LAYERS)

def make_onehot_lstm():
    return OneHotSpeciesLSTMModel(
        data_keys=DATA_KEYS, species_keys=ALL_SPECIES, unknown='zero',
        feature_statistics=FEATURE_STATS,
        hidden_size=BEST_HP['hidden_size'], num_layers=NUM_LAYERS)

def make_phylo_lstm():
    return PhylogeneticLSTMModel(
        data_keys=DATA_KEYS, species_keys=phylo_keys, mds_coords=mds_coords, unknown='zero',
        feature_statistics=FEATURE_STATS,
        hidden_size=BEST_HP['hidden_size'], num_layers=NUM_LAYERS)

def train_lstm(model, train_ds):
    # num_epochs is the tuned upper bound; _fit_torch early-stops on a held-out
    # validation slice (and skips it when the training set is too small).
    return _fit_torch(model, train_ds,
                      num_epochs=BEST_HP['num_epochs'],
                      batch_size=BEST_HP['batch_size'],
                      lr=BEST_HP['learning_rate'])

def fit_or_load(run_name, cls, train_thunk):
    if not FORCE_RETRAIN:
        try:
            return cls.load(run_name)[0]
        except ModelException:
            pass
    print(f'  training {run_name} ...')
    model = train_thunk()
    model.save(run_name)
    return model

## Experiment 1 — every species has plenty of history

A straightforward year split. The per-species mean and per-species LSTM are fit on each
species in isolation; the two multi-species LSTMs are fit once on the whole pool. Everyone is
scored on the same held-out recent years.

In [ ]:
def run_year_split():
    recs = []
    for sp in YEAR_SPECIES:
        lbl, tr = SPECIES_LABEL[sp], pool_train.select_species(sp)
        te = pool_test.select_species(sp)
        mean_m = fit_or_load(f'ms_year_mean_{lbl}_s{SEED}', MeanModel,
                             lambda tr=tr: MeanModel.fit(target_fn=target_fn, dataset=tr)[0])
        lstm_m = fit_or_load(f'ms_year_perlstm_{lbl}_s{SEED}', LSTMModel,
                             lambda tr=tr: train_lstm(make_per_species_lstm(), tr))
        recs.append(dict(model='per-species mean', species=lbl, mae=evaluate(mean_m, te)['macro']))
        recs.append(dict(model='per-species LSTM', species=lbl, mae=evaluate(lstm_m, te)['macro']))

    oh = fit_or_load(f'ms_year_onehot_s{SEED}', OneHotSpeciesLSTMModel,
                     lambda: train_lstm(make_onehot_lstm(), pool_train))
    ph = fit_or_load(f'ms_year_phylo_s{SEED}', PhylogeneticLSTMModel,
                     lambda: train_lstm(make_phylo_lstm(), pool_train))
    for name, m in [('multi one-hot', oh), ('multi phylo', ph)]:
        for sp, v in evaluate(m, pool_test)['per_species'].items():
            recs.append(dict(model=name, species=SPECIES_LABEL[sp], mae=v))
    return pd.DataFrame(recs)

year_df  = run_year_split()
year_tbl = year_df.pivot_table(index='species', columns='model', values='mae')
year_tbl.loc['— macro mean —'] = year_tbl.mean()
year_tbl.round(2)

In [ ]:
macro_year = year_tbl.drop(index='— macro mean —').mean().sort_values()
ax = macro_year.plot.bar(figsize=(6, 3), color='steelblue')
ax.set_ylabel('macro-MAE (days)'); ax.set_xlabel('')
ax.set_title('Year split — error averaged over species')
plt.xticks(rotation=20, ha='right'); plt.tight_layout(); plt.show()
macro_year.round(2)

## Experiment 2 — a newly added species

Now we hold out one species at a time. The multi-species models train on every other species
in full plus only `x` records of the held-out one; the per-species baselines get just those
`x` records. We test on the rest of the held-out species and sweep `x` from zero upward,
averaging the curves over every held-out species. At `x = 0` the one-hot model has a slot for
the newcomer that was never switched on during training, whereas the phylogenetic model can
still place it next to its relatives — a genuine zero-shot guess.

In [ ]:
def fewshot_split(sp, x):
    sp_ix = sorted(full.select_species(sp).iter_index())
    rng   = np.random.default_rng(SEED * 1000 + ALL_SPECIES.index(sp) * 100 + x)
    perm  = [sp_ix[i] for i in rng.permutation(len(sp_ix))]
    return perm[:x], perm[x:]

def run_fewshot():
    recs = []
    others = {sp: list(full.select_species(sp).iter_index()) for sp in ALL_SPECIES}
    for h in ALL_SPECIES:
        hlbl = SPECIES_LABEL[h]
        for x in X_GRID:
            tr_ix, te_ix = fewshot_split(h, x)
            if not te_ix:
                continue
            test_ds  = full.select_by_ixs(te_ix)
            pool_ix  = [ix for sp in ALL_SPECIES if sp != h for ix in others[sp]] + tr_ix
            train_ds = full.select_by_ixs(pool_ix)

            oh = fit_or_load(f'ms_fs_onehot_{hlbl}_x{x}_s{SEED}', OneHotSpeciesLSTMModel,
                             lambda train_ds=train_ds: train_lstm(make_onehot_lstm(), train_ds))
            ph = fit_or_load(f'ms_fs_phylo_{hlbl}_x{x}_s{SEED}', PhylogeneticLSTMModel,
                             lambda train_ds=train_ds: train_lstm(make_phylo_lstm(), train_ds))
            row = dict(species=hlbl, x=x,
                       **{'multi one-hot': evaluate(oh, test_ds)['macro'],
                          'multi phylo':   evaluate(ph, test_ds)['macro']})
            if x > 0:
                hd = full.select_by_ixs(tr_ix)
                mean_m = fit_or_load(f'ms_fs_mean_{hlbl}_x{x}_s{SEED}', MeanModel,
                                     lambda hd=hd: MeanModel.fit(target_fn=target_fn, dataset=hd)[0])
                row['per-species mean'] = evaluate(mean_m, test_ds)['macro']
                lstm_m = fit_or_load(f'ms_fs_perlstm_{hlbl}_x{x}_s{SEED}', LSTMModel,
                                     lambda hd=hd: train_lstm(make_per_species_lstm(), hd))
                row['per-species LSTM'] = evaluate(lstm_m, test_ds)['macro']
            recs.append(row)
            print(f'  held out {hlbl:24s} x={x:<3d} done')
    return pd.DataFrame(recs)

fewshot_df    = run_fewshot()
fewshot_curve = fewshot_df.groupby('x').mean(numeric_only=True)
fewshot_curve.round(2)

In [ ]:
ax = fewshot_curve.plot(marker='o', figsize=(6.5, 4))
ax.set_xlabel('labelled records of the held-out species (x)')
ax.set_ylabel('macro-MAE over held-out species (days)')
ax.set_title('Adapting to an unseen species'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## What the comparison shows, and where it is limited

Read the two figures together. The year split asks whether pooling helps when data is
abundant; the few-shot curve asks whether it helps when it matters most — for a species with
almost no records. A multi-species model earns its keep if it matches the per-species models
on the data-rich year split *and* pulls clearly ahead at small `x`, with the phylogenetic
variant ideally beating the one-hot variant near `x = 0`, where only relatedness can guide a
prediction for a species the model has never trained on.

**How the species signal enters the network.** In both conditioned models the species vector
is concatenated to the LSTM's *output* and feeds only the final, time-independent readout; it
never enters the recurrence. The temperature-and-daylight sequence is therefore encoded
identically for every species, and only the last step is species-aware. But species differ
mainly in *how* they integrate warmth and chill over the season — their base temperatures and
chilling needs — which is exactly what the recurrence captures. Conditioning only the readout
can shift or rescale a prediction but cannot give a species its own dynamics, so it likely
caps how much pooling can help. Feeding the species vector into the per-timestep input, or
using it to initialise the hidden state, would let it shape the dynamics themselves and is the
natural next thing to try.

**Caveats.** The phylogenetic embedding keeps only the few dimensions that explain most of the
branch-length structure among these closely related species; a more diverse species set would
need more. All LSTM variants share one hyperparameter configuration chosen by a single small
search, which keeps the experiment affordable but means none of them is individually tuned.
And these are single-seed results — every model is saved, so repeating the run under more
seeds is cheap and is the right way to attach error bars before drawing firm conclusions.